# Random-Forest-Klassifikation

In diesem Notebook wird zunächst ein Random Forest mit weitgehend unveränderten Standardparametern als Basismodell untersucht. Anschließend werden ausgewählte Hyperparameter mithilfe einer randomisierten Suche optimiert.

Die Modelle werden anhand einer fünffachen geschichteten Kreuzvalidierung bewertet. Aufgrund der ungleichen Zielklassen dient die PR-AUC als primäres Auswahlkriterium. Zusätzlich werden ROC-AUC, Precision, Recall und F1-Score betrachtet. Der unabhängige Testdatensatz wird während der Modellentwicklung nicht verwendet.

In [19]:
from pathlib import Path 
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate , train_test_split , RandomizedSearchCV
from sklearn.metrics import (
    make_scorer,
    precision_score,
    recall_score,
    f1_score
)


In [2]:
project_root = Path.cwd()
processed_dir = project_root / "data" / "processed"

In [3]:
train_data = pd.read_parquet(
    processed_dir / "train.parquet"
)

test_data = pd.read_parquet(
    processed_dir / "test.parquet"
)

print("Trainingsdaten:", train_data.shape)
print("Testdaten:", test_data.shape)

Trainingsdaten: (160000, 201)
Testdaten: (40000, 201)


In [4]:
X_train = train_data.drop(columns=["target"])
y_train = train_data["target"]

X_test = test_data.drop(columns=["target"])
y_test = test_data["target"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (160000, 200)
y_train: (160000,)
X_test: (40000, 200)
y_test: (40000,)


## 1. Random-Forest-Basismodell

Zunächst wird ein Random Forest mit weitgehend unveränderten
Standardparametern mittels geschichteter 5-Fold-Cross-Validation bewertet.
Diese Baseline zeigt, welche Leistung das Modell ohne Hyperparameteroptimierung
erreicht.

In [5]:
random_forest_baseline = RandomForestClassifier(
    random_state = 42,
    n_jobs = 4
)

cv = StratifiedKFold(
    n_splits= 5,
    shuffle = True,
    random_state=42
)

In [6]:
scoring = {
    "PR_AUC" : "average_precision" ,
    "ROC_AUC": "roc_auc" ,
    "Precision" : make_scorer(
        precision_score,
        zero_division=0 
    ),
    "Recall" : make_scorer(
        recall_score,
        zero_division=0 
    ),
    "F1": make_scorer(
        f1_score,
        zero_division=0
    )
}

In [7]:
rf_baseline_results = cross_validate(
    random_forest_baseline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=4
)

for metric in scoring:
    scores = rf_baseline_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} "
        f"(± {scores.std():.4f})"
    )

print(
    f"Mittlere Trainingszeit: "
    f"{rf_baseline_results['fit_time'].mean():.2f} Sekunden"
)

PR_AUC: 0.4066 (± 0.0063)
ROC_AUC: 0.8206 (± 0.0035)
Precision: 0.6000 (± 0.4899)
Recall: 0.0002 (± 0.0002)
F1: 0.0005 (± 0.0005)
Mittlere Trainingszeit: 1211.14 Sekunden


Ergebnis des Random-Forest-Basismodells

Der Random Forest erreicht in der geschichteten 5-Fold-Cross-Validation eine mittlere PR-AUC von rund 0,407 und eine ROC-AUC von rund 0,821. Damit liegt das Modell deutlich über der Dummy-Baseline mit einer PR-AUC von etwa 0,101 und einer ROC-AUC von 0,500. Die geringen Standardabweichungen von rund 0,006 beziehungsweise 0,004 zeigen, dass die Rangordnungsleistung zwischen den fünf Folds relativ stabil ist.

Beim Standardschwellenwert von 0,5 beträgt der Recall dagegen lediglich rund 0,0002 und der F1-Score rund 0,0005. Das Modell sagt somit nahezu keine Beobachtungen als positive Klasse voraus. Die stark schwankende Precision von durchschnittlich 0,600 bei einer Standardabweichung von rund 0,490 ist deshalb nur eingeschränkt interpretierbar: Wenn nur sehr wenige positive Vorhersagen erzeugt werden, können bereits einzelne richtige oder falsche Klassifikationen den Wert stark verändern.

## 2. Hyperparameteroptimierung

Die Hyperparametersuche wird aus Laufzeitgründen auf einer geschichteten Stichprobe von 60.000 Trainingsbeobachtungen durchgeführt. Untersucht werden unter anderem die maximale Baumtiefe, die Anzahl der je Aufteilung berücksichtigten Features, die Mindestgröße der Blätter sowie unterschiedliche Gewichtungen der Zielklassen.

Die Auswahl der besten Konfiguration erfolgt anhand der mittleren PR-AUC der Kreuzvalidierung.

In [8]:
X_split , _ , y_split, _ = train_test_split(
    X_train,
    y_train,
    train_size = 60_000,
    stratify = y_train,
    random_state = 42
)

print("Tuning-Features:", X_split.shape)
print("Tuning-Target:", y_split.shape)
print(y_split.value_counts(normalize = True))

Tuning-Features: (60000, 200)
Tuning-Target: (60000,)
target
0    0.899517
1    0.100483
Name: proportion, dtype: float64


In [9]:
param_distributions = {
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", 0.25, 0.5],
    "class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ],
    "max_samples": [None, 0.6, 0.8]
}

In [10]:
from sklearn.model_selection import RandomizedSearchCV

rf_for_search = RandomForestClassifier(
    n_estimators=100,
    bootstrap=True,
    random_state=42,
    n_jobs=4
)

search_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=rf_for_search,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="average_precision",
    cv=search_cv,
    random_state=42,
    n_jobs=4,
    verbose=2,
    return_train_score=True
)

In [11]:
random_search.fit(X_split, y_split)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV] END class_weight=None, max_depth=40, max_features=log2, max_samples=0.8, min_samples_leaf=10, min_samples_split=2; total time= 1.9min
[CV] END class_weight=None, max_depth=40, max_features=log2, max_samples=0.8, min_samples_leaf=10, min_samples_split=2; total time= 2.0min
[CV] END class_weight=None, max_depth=40, max_features=log2, max_samples=0.8, min_samples_leaf=10, min_samples_split=2; total time= 2.0min
[CV] END class_weight=None, max_depth=40, max_features=log2, max_samples=0.8, min_samples_leaf=10, min_samples_split=2; total time= 2.0min
[CV] END class_weight=None, max_depth=40, max_features=log2, max_samples=0.8, min_samples_leaf=10, min_samples_split=2; total time= 2.0min
[CV] END class_weight=balanced, max_depth=10, max_features=0.25, max_samples=0.8, min_samples_leaf=10, min_samples_split=10; total time= 6.3min
[CV] END class_weight=balanced, max_depth=10, max_features=0.25, max_samples=0.8, min_samples_leaf=1

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'class_weight': [None, 'balanced', ...], 'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2', ...], 'max_samples': [None, 0.6, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",4
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strateg

In [12]:
print(
    "Beste mittlere Tuning-PR-AUC:",
    round(random_search.best_score_, 4)
)

search_results = (
    pd.DataFrame(random_search.cv_results_)
    .sort_values("rank_test_score")
)

columns_to_show = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "param_max_depth",
    "param_min_samples_split",
    "param_min_samples_leaf",
    "param_max_features",
    "param_class_weight",
    "param_max_samples",
    "mean_fit_time"
]

search_results[columns_to_show].head(10)

Beste mittlere Tuning-PR-AUC: 0.4564


,rank_test_score,mean_test_score,std_test_score,param_max_depth,param_min_samples_split,param_min_samples_leaf,param_max_features,param_class_weight,param_max_samples,mean_fit_time
0,1,0.456393,0.013934,40,2,10,log2,NaN,0.8,119.084389
6,2,0.436578,0.013223,20,10,1,log2,NaN,0.8,93.008734
13,3,0.402359,0.009475,40,5,1,sqrt,NaN,None,250.694260
11,4,0.380405,0.012818,None,10,1,0.25,NaN,0.8,3593.127581
5,5,0.364816,0.005710,10,5,5,log2,balanced_subsample,None,63.339502
8,6,0.348470,0.012922,10,10,5,0.25,NaN,0.8,367.414611
7,7,0.343248,0.006910,10,10,2,log2,balanced,0.8,52.919883
14,8,0.342522,0.006587,20,20,2,log2,balanced_subsample,0.8,65.564315
10,9,0.337860,0.005174,10,20,2,sqrt,balanced_subsample,0.6,85.327762
9,10,0.323154,0.006133,20,10,5,0.25,balanced,0.8,515.150190


## Validierung der besten Hyperparameter

Die auf der Tuning-Stichprobe gefundene Parameterkonfiguration wird nun mit
derselben geschichteten 5-Fold-Cross-Validation wie das Basismodell auf allen
160.000 Trainingsbeobachtungen bewertet. Dadurch kann beurteilt werden, ob
die Verbesserung auch außerhalb der kleineren Tuning-Stichprobe bestehen
bleibt. Der Testdatensatz wird weiterhin nicht verwendet.

In [13]:
best_rf_params = random_search.best_params_

tuned_random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=4,
    **best_rf_params
)

full_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [14]:
tuned_rf_results = cross_validate(
    tuned_random_forest,
    X_train,
    y_train,
    cv=full_cv,
    scoring=scoring,
    n_jobs=4
)

for metric in scoring:
    scores = tuned_rf_results[f"test_{metric}"]

    print(
        f"{metric}: "
        f"{scores.mean():.4f} "
        f"(± {scores.std():.4f})"
    )

print(
    "Mittlere Trainingszeit: "
    f"{tuned_rf_results['fit_time'].mean():.2f} Sekunden"
)

PR_AUC: 0.4751 (± 0.0077)
ROC_AUC: 0.8481 (± 0.0021)
Precision: 0.0000 (± 0.0000)
Recall: 0.0000 (± 0.0000)
F1: 0.0000 (± 0.0000)
Mittlere Trainingszeit: 230.64 Sekunden


### Ergebnis des optimierten Random Forest

Die optimierte Parameterkonfiguration verbessert die mittlere PR-AUC von
0,4066 auf 0,4751 und die ROC-AUC von 0,8206 auf 0,8481. Gleichzeitig sinkt
die mittlere Trainingszeit eines Fold-Modells von rund 998 auf 328 Sekunden.
Die Hyperparameteroptimierung führt somit sowohl zu einer besseren
Rangordnungsleistung als auch zu einem effizienteren Modell.

Beim Standardschwellenwert von 0,5 werden jedoch keine positiven Fälle
vorhergesagt, sodass Precision, Recall und F1 den Wert null annehmen. Da die
Optimierung anhand der schwellenwertunabhängigen PR-AUC erfolgte, widerspricht
dies nicht der verbesserten Rangordnungsleistung. Eine geeignete
Schwellenwertwahl wird erst nach dem Vergleich aller Modellkandidaten für das
ausgewählte Modell durchgeführt.

# Fold-Ergebnisse speichern

In [15]:
results_dir = project_root / "results" / "cv"
results_dir.mkdir(parents=True, exist_ok=True)

n_folds = len(tuned_rf_results["test_PR_AUC"])

tuned_rf_scores = pd.DataFrame({
    "Modell": ["RandomForest_Tuned"] * n_folds,
    "Fold": range(1, n_folds + 1),
    "PR_AUC": tuned_rf_results["test_PR_AUC"],
    "ROC_AUC": tuned_rf_results["test_ROC_AUC"],
    "Precision": tuned_rf_results["test_Precision"],
    "Recall": tuned_rf_results["test_Recall"],
    "F1": tuned_rf_results["test_F1"],
    "Fit_Time": tuned_rf_results["fit_time"],
    "Score_Time": tuned_rf_results["score_time"]
})

tuned_rf_scores.to_csv(
    results_dir / "random_forest_tuned_cv.csv",
    index=False
)

tuned_rf_scores

,Modell,Fold,PR_AUC,ROC_AUC,Precision,Recall,F1,Fit_Time,Score_Time
0,RandomForest_Tuned,1,0.468786,0.846554,0.0,0.0,0.0,272.133814,2.194315
1,RandomForest_Tuned,2,0.471638,0.848219,0.0,0.0,0.0,265.418379,2.702503
2,RandomForest_Tuned,3,0.485704,0.851974,0.0,0.0,0.0,264.408942,2.918055
3,RandomForest_Tuned,4,0.466616,0.845754,0.0,0.0,0.0,272.310952,2.103722
4,RandomForest_Tuned,5,0.482668,0.847975,0.0,0.0,0.0,78.904000,0.654729


In [16]:
n_folds = len(rf_baseline_results["test_PR_AUC"])

rf_baseline_scores = pd.DataFrame({
    "Modell": ["RandomForest_Baseline"] * n_folds,
    "Fold": range(1, n_folds + 1),
    "PR_AUC": rf_baseline_results["test_PR_AUC"],
    "ROC_AUC": rf_baseline_results["test_ROC_AUC"],
    "Precision": rf_baseline_results["test_Precision"],
    "Recall": rf_baseline_results["test_Recall"],
    "F1": rf_baseline_results["test_F1"],
    "Fit_Time": rf_baseline_results["fit_time"],
    "Score_Time": rf_baseline_results["score_time"]
})

rf_baseline_scores.to_csv(
    results_dir / "random_forest_baseline_cv.csv",
    index=False
)

In [17]:
tuning_dir = project_root / "results" / "tuning"
tuning_dir.mkdir(parents=True, exist_ok=True)

search_results.to_csv(
    tuning_dir / "random_forest_random_search.csv",
    index=False
)

In [18]:
rf_comparison = pd.concat(
    [
        rf_baseline_scores,
        tuned_rf_scores
    ],
    ignore_index=True
)

rf_comparison.groupby("Modell")[
    [
        "PR_AUC",
        "ROC_AUC",
        "Precision",
        "Recall",
        "F1",
        "Fit_Time"
    ]
].agg(["mean", "std"])

PR_AUC             ROC_AUC           Precision  \
                           mean       std      mean       std      mean   
Modell                                                                    
RandomForest_Baseline  0.406620  0.007098  0.820579  0.003883       0.6   
RandomForest_Tuned     0.475082  0.008566  0.848095  0.002395       0.0   

                                   Recall                 F1           \
                            std      mean      std      mean      std   
Modell                                                                  
RandomForest_Baseline  0.547723  0.000249  0.00026  0.000497  0.00052   
RandomForest_Tuned     0.000000  0.000000  0.00000  0.000000  0.00000   

                          Fit_Time              
                              mean         std  
Modell                                          
RandomForest_Baseline  1211.136053  334.641678  
RandomForest_Tuned      230.635217   84.899787